<a href="https://colab.research.google.com/github/jiuwong/sfu_AppliedAI_DataAnalytics/blob/main/7_1_hyperparameter_tuning_using_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://sfudial.ca/wp-content/uploads/SFU-DIAL-Logo.png" width=40%>&nbsp;&nbsp;&nbsp;&nbsp;<img src="https://www.sfu.ca/content/dam/sfu/images/brand_extension/SFU-Big-Data_Logo.png" width=40%>

# Lab 7: From Hyperparameter Tuning to Prompt Tuning

This lab explores the fascinating parallels and distinctions between traditional machine learning hyperparameter tuning and the emerging field of prompt tuning for Large Language Models (LLMs).

## Part 1: Classic ML vs Prompt Tuning

In traditional machine learning, you adjust numerical parameters of your models. In the world of LLMs, your *words* — the prompt's phrasing, structure, and examples — become your 'hyperparameters'!

Here is a comparison of tuning in classic ML and prompt tuning for LLMs

In [ ]:
import pandas as pd

# Check if running in Colab or Jupyter
try:
    from IPython.display import display
    IN_NOTEBOOK = True
except ImportError:
    IN_NOTEBOOK = False
    # Fallback for non-notebook environments
    def display(obj):
        print(obj)

# This DataFrame illustrates the core differences between tuning in classic ML and prompt tuning for LLMs.
comparison_table = pd.DataFrame({
    "Classical ML": [
        "Adjust numerical hyperparameters (learning rate, depth, dropout)",
        "Use metrics (accuracy, F1) to compare models",
        "Log runs in MLflow",
        "Tune for generalization",
        "Use search strategies (grid, Bayesian)"
    ],
    "Prompt Tuning (LLMs)": [
        "Adjust prompt phrasing, structure, tone, examples",
        "Compare responses (clarity, correctness, relevance)",
        "Record prompt versions + outcomes",
        "Tune for robustness across inputs",
        "Use iterative refinement and feedback"
    ]
})

display(comparison_table)

print("\n--- Continue to Part 2 to see prompt engineering best practices in action! ---")

## Part 2: Prompt Engineering Best Practices

In this section, we'll dive into practical prompt engineering techniques using an open-source Large Language Model. We'll explore how small changes in prompt design can significantly impact the quality of the model's responses.

### Option 1: Using ChatGPT or your own LLM platform

- You will copy and paste the example prompts directly into ChatGPT (or any LLM interface) to observe differences.

- You will record outcomes manually in a Google Sheet (columns: Prompt Version, Response Quality, Notes).

### Option 2: Using Colab Notebook

This option is automated but might take a while (20 mins) if you don't have GPU! We also selected a very lightweight model so the performance cannot compare to ChatGPT, but should show the difference in prompting!

### If you chose Option 1:

#### Define Prompt Engineering Examples

Below is a collection of examples demonstrating various prompt engineering best practices. For each 'best practice' (e.g., 'Be Specific', 'Provide Context'), we define two prompts: a generic 'Prompt A' and an improved 'Prompt B' that incorporates the best practice. These examples will help us compare the model's responses to different prompt styles.

- ACTION: For each example, copy both the context and the prompt to your AI chatbot, observe how responses differ between prompt A and B.

#### 1. Be Specific

**Context:**
```
Quarterly Financial Report: The company's Q2 revenue grew by 12% due to strong demand in Europe, particularly in the consumer electronics sector. Operating costs decreased by 5% after restructuring initiatives earlier in the year. North American sales remained flat, while Asia-Pacific saw moderate growth of 3%. Overall, net income increased by 9% compared to the previous quarter.
```

**Prompt A:**
```
Summarize this report.
```

**Prompt B:**
```
Summarize this report in three bullet points, each under 15 words, highlighting key business insights only.
```

#### 2. Provide Context

**Context:**
```
Project Update: The client 'BlueWave Technologies' requested delivery of the analytics dashboard one week earlier to align with an upcoming product launch. The team has completed 80% of the work and will need additional testing time.
```

**Prompt A:**
```
Write an email to a client.
```

**Prompt B:**
```
You are a project manager at a tech company. Write a friendly but professional email to a client whose project deadline moved forward by one week, explaining the reason and next steps.
```

#### 3. Set the Role Clearly

**Context:**
```
Professional Development: You recently completed a 3-day workshop on 'AI for Business Decision-Making', covering topics like automation strategy, ethical AI, and data-driven leadership. Participants shared case studies from multiple industries.
```

**Prompt A:**
```
Write a LinkedIn post about finishing a workshop.
```

**Prompt B:**
```
You are a marketing analyst who just completed a workshop on AI in business. Write a 3-sentence LinkedIn post highlighting what you learned and thanking the organizers. Use a professional yet approachable tone.
```

#### 4. Structure Complex Tasks Step-by-Step

**Context:**
```
Scenario: Your team has been struggling with inconsistent communication and missed deadlines. Some members work remotely, and meetings often run too long without clear action points. Productivity has dropped by 15% this quarter.
```

**Prompt A:**
```
Give some advice to improve team productivity.
```

**Prompt B:**
```
1. List three specific strategies to improve team productivity. 2. Explain briefly how each strategy helps. 3. End with a motivational line for the team.
```

#### 5. Define the Output Format

**Context:**
```
Customer Feedback Summary: Users have reported issues with slow app load times, confusing menu navigation, and occasional crashes after updates. Positive feedback highlights the app's clean design and helpful customer support team.
```

**Prompt A:**
```
Generate a summary of customer complaints.
```

**Prompt B:**
```
Generate a summary of customer complaints in JSON with fields: { 'top_issues': [], 'frequency_analysis': {}, 'sentiment_summary': '' }
```

### If you chose Option 2:

#### 2.1 Setup and Install Necessary Libraries

1. Go to **Runtime → Change runtime type → Hardware accelerator → GPU** (if available).
2. If you don't have GPU access, the notebook will still run on CPU using a lightweight quantized model.
3. This notebook is for educational purposes — feel free to **try the same prompts directly in ChatGPT or any LLM interface** to compare outputs.
4. Running time: ~1–2 minutes per example on GPU, ~5–8 minutes total on CPU.

First, we need to install the `transformers` library from Hugging Face, along with `accelerate` and `bitsandbytes` for efficient model loading and inference, and `sentencepiece` which is often a dependency for tokenizers.

**Note:** The `accelerate` package is required if you want to use `device_map="auto"` for automatic GPU/CPU distribution. If `accelerate` is not installed, the notebook will still work but will use a simpler device selection method.

In [ ]:
# Install required packages (uncomment if running in Colab or if packages are not installed)
!pip install -q transformers accelerate bitsandbytes sentencepiece

#### 2.2 Load an Open-Source Chat Model

We'll use the `pipeline` function from the `transformers` library to easily load a pre-trained chat model. For this workshop, we're using `TinyLlama/TinyLlama-1.1B-Chat-v1.0`, a lightweight open-source instruction-tuned model. If `accelerate` is available, `device_map="auto"` helps in automatically distributing the model layers across available devices (like GPU) for optimal performance. Otherwise, the model will automatically use GPU if available, or fall back to CPU.

This model is not as powerful as Chat-GPT but can demonstrate LLM's response to prompt tuning. As a fall back, a smaller model is used if TinyLlama cannot work on your device.

In [ ]:
# Check if transformers is available
try:
    from transformers import pipeline
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("⚠️  transformers library not available.")
    print("   Install with: pip install transformers accelerate bitsandbytes sentencepiece")
    print("   Or run this notebook in Google Colab where packages are pre-installed.")
    print("   For now, skipping model loading - you can still review the prompt examples above.")

if TRANSFORMERS_AVAILABLE:
    # Check if accelerate is available (required for device_map)
    try:
        import accelerate
        ACCELERATE_AVAILABLE = True
    except ImportError:
        ACCELERATE_AVAILABLE = False
        print("⚠️  accelerate library not available. Model will run on CPU without device_map optimization.")
        print("   Install with: pip install accelerate")

    mistral_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Using TinyLlama as requested
    fallback_model_name = "distilgpt2"

    model = None
    try:
        print(f"Attempting to load {mistral_model_name}...")
        # Only use device_map if accelerate is available, otherwise let transformers handle device selection
        # Use dtype instead of deprecated torch_dtype
        pipeline_kwargs = {"model": mistral_model_name, "dtype": "auto"}
        if ACCELERATE_AVAILABLE:
            pipeline_kwargs["device_map"] = "auto"
        else:
            # Try to use GPU if available, otherwise CPU
            import torch
            pipeline_kwargs["device"] = 0 if torch.cuda.is_available() else -1

        model = pipeline("text-generation", **pipeline_kwargs)
        print(f"Successfully loaded {mistral_model_name} for text generation.")
    except Exception as e:
        print(f"Failed to load {mistral_model_name}: {e}")
        print(f"Falling back to {fallback_model_name}...")
        # Fallback to a smaller model if loading fails
        try:
            pipeline_kwargs = {"model": fallback_model_name}
            if ACCELERATE_AVAILABLE:
                pipeline_kwargs["device_map"] = "auto"
            else:
                import torch
                pipeline_kwargs["device"] = 0 if torch.cuda.is_available() else -1

            model = pipeline("text-generation", **pipeline_kwargs)
            print(f"Successfully loaded {fallback_model_name} for text generation.")
        except Exception as e2:
            print(f"Failed to load fallback model: {e2}")
            print("⚠️  Model loading failed. This notebook requires GPU access or proper model installation.")
            print("   Consider running in Google Colab or installing required dependencies:")
            print("   pip install transformers accelerate bitsandbytes sentencepiece")
            model = None

#### 2.3 Prepare the same prompt comparison examples

In [ ]:
examples = [
    {
        "title": "Be Specific",
        "context": (
            "Quarterly Financial Report: The company's Q2 revenue grew by 12% due to strong demand in Europe, "
            "particularly in the consumer electronics sector. Operating costs decreased by 5% after restructuring "
            "initiatives earlier in the year. North American sales remained flat, while Asia-Pacific saw moderate "
            "growth of 3%. Overall, net income increased by 9% compared to the previous quarter."
        ),
        "prompt_a": "Summarize this report.",
        "prompt_b": "Summarize this report in three bullet points, each under 15 words, highlighting key business insights only."
    },
    {
        "title": "Provide Context",
        "context": (
            "Project Update: The client 'BlueWave Technologies' requested delivery of the analytics dashboard one week earlier "
            "to align with an upcoming product launch. The team has completed 80% of the work and will need additional testing time."
        ),
        "prompt_a": "Write an email to a client.",
        "prompt_b": "You are a project manager at a tech company. Write a friendly but professional email to a client whose project deadline moved forward by one week, explaining the reason and next steps."
    },
    {
        "title": "Set the Role Clearly",
        "context": (
            "Professional Development: You recently completed a 3-day workshop on 'AI for Business Decision-Making', covering "
            "topics like automation strategy, ethical AI, and data-driven leadership. Participants shared case studies from multiple industries."
        ),
        "prompt_a": "Write a LinkedIn post about finishing a workshop.",
        "prompt_b": "You are a marketing analyst who just completed a workshop on AI in business. Write a 3-sentence LinkedIn post highlighting what you learned and thanking the organizers. Use a professional yet approachable tone."
    },
    {
        "title": "Structure Complex Tasks Step-by-Step",
        "context": (
            "Scenario: Your team has been struggling with inconsistent communication and missed deadlines. Some members work remotely, "
            "and meetings often run too long without clear action points. Productivity has dropped by 15% this quarter."
        ),
        "prompt_a": "Give some advice to improve team productivity.",
        "prompt_b": "1. List three specific strategies to improve team productivity. 2. Explain briefly how each strategy helps. 3. End with a motivational line for the team."
    },
    {
        "title": "Define the Output Format",
        "context": (
            "Customer Feedback Summary: Users have reported issues with slow app load times, confusing menu navigation, and occasional crashes after updates. "
            "Positive feedback highlights the app's clean design and helpful customer support team."
        ),
        "prompt_a": "Generate a summary of customer complaints.",
        "prompt_b": ("Generate a summary of customer complaints in JSON with fields: { 'top_issues': [], "
                     "'frequency_analysis': {}, 'sentiment_summary': '' }")
    }
]

#### 2.4 Running Prompts in Colab with TinyLlama and Collect Responses

> ## **Warning: This might take a while!**

We will iterate through each example, feeding both 'Prompt A' (the less-optimized version) and 'Prompt B' (the improved version) to our TinyLlama model. We'll collect the generated responses to compare how prompt engineering influences the output.

In this line of code below, we can do some actual **hyperparameter tuning**:

```python
response = model(formatted_prompt, max_new_tokens=250, do_sample=True, temperature=0.7, return_full_text=False)[0]['generated_text']
```

- `max_new_tokens` This parameter directly controls the maximum length of the generated response. It specifies the highest number of new tokens (words or sub-word units) the model is allowed to create after processing your input prompt. If set too low, the model's output might be cut off prematurely. If set too high, it could generate very long, potentially repetitive, or irrelevant text, and consume more resources.
- `do_sample=True` When do_sample is set to True, the model samples from the probability distribution of possible next tokens. Instead of always picking the single most probable word (which can lead to very generic or repetitive text), do_sample=True allows the model to randomly select a word based on its probability. This introduces creativity and variety into the generated text. If do_sample were False (or not set, as False is often the default), the model would always choose the most probable word, leading to deterministic and less diverse outputs.
- `temperature=0.7` This parameter works hand-in-hand with do_sample=True to control the randomness and creativity of the sampling process. It's typically a value between 0 and 1 (though some implementations allow higher values).
  - Low temperature (e.g., 0.1 - 0.3): Makes the model more conservative and focused. It will be more likely to pick highly probable words, resulting in outputs that are more deterministic, factual, and less creative, but also less prone to generating nonsensical text.
  - Moderate temperature (e.g., 0.5 - 0.7): Offers a good balance between coherence and creativity, often producing diverse and interesting text while generally remaining on topic.
  - High temperature (e.g., 0.8 - 1.0+): Increases the model's willingness to pick less probable words, leading to more random, creative, and sometimes surprising outputs. However, very high temperatures can make the text less coherent, less factual, or even nonsensical.

In [ ]:
results = []

if model is not None and TRANSFORMERS_AVAILABLE:
    # Get the tokenizer from the pipeline to apply the chat template correctly
    tokenizer = model.tokenizer

    for ex in examples:
        print(f"\n### ⚙∇ {ex['title']}")
        if ex.get('context'):  # Print context if available
            print(f"Context: {ex['context']}")
        print(f"Prompt A: {ex['prompt_a']}")
        print(f"Prompt B: {ex['prompt_b']}")

        for label, prompt in zip(['A', 'B'], [ex['prompt_a'], ex['prompt_b']]):
            # Combine context and prompt
            combined_prompt_content = f"Context: {ex['context']}\n\n{prompt}" if ex.get('context') else prompt

            # Format the prompt using the tokenizer's chat template
            messages = [{"role": "user", "content": combined_prompt_content}]
            formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

            # Generate response from the model, ensuring only new text is returned
            # Increased max_new_tokens slightly for potentially longer responses
            try:
                response = model(formatted_prompt, max_new_tokens=250, do_sample=True, temperature=0.7, return_full_text=False)[0]['generated_text']
                results.append({"example": ex['title'], "prompt": f"Prompt {label}", "output": response})
                print(f"Output for Prompt {label}:\n{response}\n")
            except Exception as e:
                print(f"⚠️  Error generating response for Prompt {label}: {e}")
                results.append({"example": ex['title'], "prompt": f"Prompt {label}", "output": f"Error: {str(e)}"})
        print("--- Responses generated for this example. --- ")
else:
    print("⚠️  Model not available. Skipping prompt execution.")
    print("   To run this section:")
    print("   1. Install transformers: pip install transformers accelerate bitsandbytes sentencepiece")
    print("   2. Or run this notebook in Google Colab")
    print("   3. Or use Option 1 above to test prompts manually in ChatGPT")

#### 2.5 Display Generated Results

After running all the prompts, we'll compile the results into a pandas DataFrame. This allows for a clear side-by-side comparison of the outputs from 'Prompt A' and 'Prompt B' for each prompt engineering best practice, making it easier to analyze the impact of prompt improvements.

In [ ]:
if results:
    results_df = pd.DataFrame(results)
    display(results_df)
else:
    print("No results to display. Run the model execution section above to generate results.")

## Part 3: Reflection Activity

Now that you've seen the impact of prompt engineering, it's your turn to apply these principles. This activity encourages you to think critically about how you can improve your own LLM interactions.

### Reflection Instructions:

1. **Choose a Task**: Select one business-related task you often perform (e.g., writing a marketing email, summarizing meeting notes, generating a report outline).
2. **Craft Prompts**: Write two versions of a prompt for this task: 'Prompt A' (a basic version) and 'Prompt B' (an improved version applying *one* specific prompt engineering best practice, like 'Be Specific' or 'Provide Context').
3. **Run and Compare**: Execute both prompts using the model loaded above (you'll need to write Python code similar to the examples). Carefully compare the outputs.
4. **Discuss**: Reflect on the following questions:
   * Which prompt produced more reliable, accurate, or useful results?
   * What 'hyperparameter' (i.e., which specific aspect of prompt engineering) did you change between Prompt A and Prompt B?
   * How would you track these changes and iterate further on your prompt design in a real-world scenario?

## Other prompt tuning examples:
[Best practices for prompt engineering with OpenAI](https://help.openai.com/en/articles/6654000-best-practices-for-prompt-engineering-with-the-openai-api)

### ✅ Optional: Log Results as CSV

To simulate experiment tracking, you can save the results of your prompt tuning experiments to a CSV file. This is a simple way to keep a record of different prompt versions and their corresponding outputs, which is crucial for iterating and improving your prompt engineering over time.

In [ ]:
# Saves the DataFrame containing all prompt results to a CSV file.
# This CSV can serve as a simple log of your prompt tuning experiments.
if results:
    results_df.to_csv("prompt_tuning_results.csv", index=False)
    print("Results saved to prompt_tuning_results.csv")
else:
    print("No results to save. Run the model execution section above to generate results.")